# Bike Store Relational Database Project
**Goal:** Analyze sales, inventory, and staff performance for a fictional bike retail company using SQL.

In [ ]:
%load_ext sql
%sql sqlite:///BikeStore.db

### 1. What are the top 10 most expensive products we sell?

In [ ]:
%%sql

SELECT product_name, list_price 
FROM products 
ORDER BY list_price DESC 
LIMIT 10;

### 2. How many customers do we have in the state of New York (NY)?

In [ ]:
%%sql

SELECT count(*) AS total_ny_customers
FROM customers 
WHERE state = 'NY';

### 3. How many products do we have in each category?

In [ ]:
%%sql

SELECT c.category_name, COUNT(p.product_id) AS total_products
FROM categories c
JOIN products p ON c.category_id = p.category_id
GROUP BY c.category_name
ORDER BY total_products DESC;

### 4. Which brands generate the most revenue?

In [ ]:
%%sql

SELECT b.brand_name, 
       ROUND(SUM(oi.quantity * oi.list_price * (1 - oi.discount)), 2) AS total_revenue
FROM brands b
JOIN products p ON b.brand_id = p.brand_id
JOIN order_items oi ON p.product_id = oi.product_id
GROUP BY b.brand_name
ORDER BY total_revenue DESC;

### 5. Who are our top 5 most valuable customers?

In [ ]:
%%sql

SELECT c.first_name, c.last_name, 
       ROUND(SUM(oi.quantity * oi.list_price * (1 - oi.discount)), 2) AS total_spent
FROM customers c
JOIN orders o ON c.customer_id = o.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY c.customer_id
ORDER BY total_spent DESC
LIMIT 5;

### 6. Which store has the most orders, and who is the best-performing staff member there?

In [ ]:
%%sql

SELECT s.store_name, st.first_name || ' ' || st.last_name AS staff_name, COUNT(o.order_id) AS total_orders
FROM stores s
JOIN orders o ON s.store_id = o.store_id
JOIN staffs st ON o.staff_id = st.staff_id
GROUP BY s.store_name, staff_name
ORDER BY total_orders DESC;

### 7. Shipping Efficiency: Which store has the highest number of late shipments?

In [ ]:
%%sql

SELECT 
    s.store_name, 
    COUNT(o.order_id) AS late_shipments
FROM orders o
JOIN stores s ON o.store_id = s.store_id
WHERE o.shipped_date > o.required_date
GROUP BY s.store_name
ORDER BY late_shipments DESC
LIMIT 1;

### 8. Inventory Alert: Which historically high-revenue products are currently out of stock?

In [ ]:
%%sql

WITH ProductRevenue AS (
    SELECT product_id, ROUND(SUM(quantity * list_price * (1 - discount)), 2) AS total_revenue
    FROM order_items
    GROUP BY product_id
),
ProductStock AS (
    SELECT product_id, SUM(quantity) AS total_stock
    FROM stocks
    GROUP BY product_id
)
SELECT 
    p.product_name, 
    pr.total_revenue, 
    ps.total_stock
FROM products p
JOIN ProductRevenue pr ON p.product_id = pr.product_id
JOIN ProductStock ps ON p.product_id = ps.product_id
WHERE ps.total_stock = 0
ORDER BY pr.total_revenue DESC
LIMIT 5;

### 9. Staff Hierarchy & Performance: How much revenue has each employee generated?

In [ ]:
%%sql

SELECT 
    e.first_name || ' ' || e.last_name AS employee_name,
    s.store_name,
    m.first_name || ' ' || m.last_name AS manager_name,
    ROUND(SUM(oi.quantity * oi.list_price * (1 - oi.discount)), 2) AS total_revenue
FROM staffs e
LEFT JOIN staffs m ON e.manager_id = m.staff_id
JOIN stores s ON e.store_id = s.store_id
JOIN orders o ON e.staff_id = o.staff_id
JOIN order_items oi ON o.order_id = oi.order_id
GROUP BY e.staff_id, employee_name, s.store_name, manager_name
ORDER BY total_revenue DESC;

### 10. Regional Market Analysis: What is the most popular category of bicycle in each state?

In [ ]:
%%sql

WITH CategorySales AS (
    SELECT 
        cu.state,
        cat.category_name,
        SUM(oi.quantity) AS total_quantity_sold
    FROM customers cu
    JOIN orders o ON cu.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    JOIN products p ON oi.product_id = p.product_id
    JOIN categories cat ON p.category_id = cat.category_id
    GROUP BY cu.state, cat.category_name
),
RankedSales AS (
    SELECT 
        state,
        category_name,
        total_quantity_sold,
        RANK() OVER(PARTITION BY state ORDER BY total_quantity_sold DESC) as rank
    FROM CategorySales
)
SELECT state, category_name, total_quantity_sold
FROM RankedSales
WHERE rank = 1;